### 🧪 CodeCritic Tool Provider Evaluation — Summary, Return Codes & Violation Metrics

This notebook analyzes tool-level execution logs from the **CodeCritic provider log archive**, focusing on **validation outcomes**, **performance patterns**, and **violation density** across tools and runs. It transforms raw JSON output into structured metrics, enabling a suite of diagnostic visualizations.

#### 🔍 Overview

* **Summary Message Distribution**
  Bar chart showing how frequently each `summary` type (including emoji) is produced by tools.

* **Return Code Analysis**
  Split-panel visualization of pass/fail frequency and corresponding latency distributions.

* **Violation Count Metrics**
  Mean number of violations per tool per run, helping isolate noisy or unstable providers.

* **Latency Distributions**
  Box plot with jittered points illustrating runtime variability across tools.

* **Tool Usage Frequency**
  Count of how often each tool provider is invoked across all runs.

This notebook supports regression tracking, stability comparisons, and performance debugging of tool providers within the CodeCritic agent framework.

### 🧰 Notebook Setup: Tool Output Parsing and Preprocessing

This initialization cell sets up the working environment and loads **raw tool provider logs** into a clean, analysis-ready DataFrame (`df_tool_expanded`).

#### 🧠 What this code does:

* **Sets up the project environment:**

  * Navigates to the root of the CodeCritic repo.
  * Adds it to the Python path so that internal imports work correctly.

* **Loads and filters tool logs:**

  * Reads the `provider_logs.csv` backup file from the `tests/backup/` directory.
  * Filters to include only rows where `provider_type == PROVIDER_TYPE.TOOL`.

* **Parses and expands JSON output fields:**

  * Safely deserializes the `output` JSON string into Python dicts.
  * Extracts individual structured fields:

    * `return_code`, `summary`, `stderr`, `stdout`, `violations`
    * Nested `metrics`: `violation_count`, `error_count`, `files_checked`, `files_reformatted`, `lines_changed`

* **Handles flexible field formats:**

  * Supports both nested JSON (e.g., `metrics["violation_count"]`) and flat keys (e.g., `metrics.violation_count`) in case of logging variations.
  * Ensures safe fallback behavior for missing or malformed data.

* **Produces `df_tool_expanded`:**

  * A clean and analysis-ready DataFrame used throughout the rest of the notebook.

#### 🔍 What you're observing:

* This preprocessing block is critical for ensuring downstream visualizations work with **clean, structured, and aligned data**.
* It bridges the raw FSM output with aggregated insights by unpacking the operational results of tool providers.
* Any visualization that follows will assume this transformation has occurred.


In [1]:
import os
import sys
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

BACKUP_PATH = Path("tests/backup/provider_logs.csv")
assert BACKUP_PATH.exists(), "❌ Backup file not found"

df = pd.read_csv(BACKUP_PATH)

df_tool = df[df["provider_type"] == "PROVIDER_TYPE.TOOL"].copy()

def safe_json_parse(val):
    try:
        return json.loads(val) if isinstance(val, str) else {}
    except Exception:
        return {}

df_tool["output_parsed"] = df_tool["output"].apply(safe_json_parse)
df_tool["return_code"] = df_tool["output_parsed"].apply(lambda x: x.get("return_code"))
df_tool["summary"] = df_tool["output_parsed"].apply(lambda x: x.get("summary"))
df_tool["stderr"] = df_tool["output_parsed"].apply(lambda x: x.get("stderr"))
df_tool["stdout"] = df_tool["output_parsed"].apply(lambda x: x.get("stdout"))
df_tool["violations"] = df_tool["output_parsed"].apply(lambda x: x.get("violations"))

df_tool["violation_count"] = df_tool["output_parsed"].apply(
    lambda x: x.get("metrics", {}).get("violation_count") 
              or x.get("metrics.violation_count")
)
df_tool["error_count"] = df_tool["output_parsed"].apply(
    lambda x: x.get("metrics", {}).get("error_count") 
              or x.get("metrics.error_count")
)
df_tool["files_checked"] = df_tool["output_parsed"].apply(
    lambda x: x.get("metrics", {}).get("files_checked") 
              or x.get("metrics.files_checked")
)
df_tool["files_reformatted"] = df_tool["output_parsed"].apply(
    lambda x: x.get("metrics", {}).get("files_reformatted") 
              or x.get("metrics.files_reformatted")
)
df_tool["lines_changed"] = df_tool["output_parsed"].apply(
    lambda x: x.get("metrics", {}).get("lines_changed") 
              or x.get("metrics.lines_changed")
)

df_tool_expanded = df_tool.drop(columns=["output_parsed"])


### 🧮 Usage by Tool Provider

This cell visualizes **how frequently each tool was used** across the dataset, helping highlight both popular and rarely invoked tools in a clean, annotated bar chart.

#### 🧠 What this code does:

* Maps raw `provider_id` values to readable `tool_name`s for display.
* Aggregates the total number of times each tool was invoked.
* Sorts tools by ascending usage for a clearer left-to-right trend.
* Creates a Plotly bar chart with:

  * Tool names on the x-axis
  * Usage count on the y-axis
  * Color-coded bars using the `dense` sequential palette
  * Value annotations positioned above each bar

#### 🔍 What you're observing:

* Tools on the right are the **most frequently used**, while those on the left may be:

  * Rarely applicable
  * Newly introduced
  * Deprecated or failing early
* Great for identifying **primary tools** vs. experimental ones.
* Also useful for confirming load balancing or sampling coverage across tool variants.

This cell acts as a **quantitative index** of how often each tool contributes to FSM operations or workflow steps.

In [2]:
import plotly.express as px
from app.enums.tool_enums import ToolProvider

# Map provider_id to name
df_tool_expanded['tool_name'] = df_tool_expanded['provider_id'].apply(ToolProvider.get_name)

# Aggregate usage counts
tool_counts = df_tool_expanded['tool_name'].value_counts().sort_values()

# Build Plotly figure
fig = px.bar(
    x=tool_counts.index,
    y=tool_counts.values,
    color=tool_counts.index,
    color_discrete_sequence=px.colors.sequential.dense,
    labels={'x': 'Tool Provider', 'y': 'Usage Count'},
    text=tool_counts.values,
    title='Usage by Tool Provider',
)

# Update layout for spacing and style
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title="Tool Provider",
    yaxis_title="Usage Count",
    height=400,
    width=1500,
    showlegend=False,
    margin=dict(t=60, b=40),
    plot_bgcolor='white'
)
fig.update_yaxes(range=[0, max(tool_counts.values) * 1.2])

fig.show()


### ⏱️ Tool Invocation Frequency (Grouped by 60-Second Intervals)

This cell visualizes **how frequently each tool is invoked over time**, grouped into 60-second bins. It provides a timeline-style view of tool activity, helping identify bursts, patterns, or idle periods during execution.

#### 🧠 What this code does:

* **Binning**:

  * Floors each tool execution timestamp into 60-second intervals (`time_bin`) for time-based grouping.

* **Aggregation**:

  * Groups by both `tool_name` and `time_bin`, counting how many times each tool is called per interval.

* **Plotting**:

  * Uses a grouped bar chart to display each tool’s invocation count in each 60-second window.
  * Bars are color-coded by tool for easy comparison.
  * Adds numeric annotations above each bar (`text_auto=True`).

#### 🔍 What you're observing:

* This plot reveals **temporal distribution of tool usage**:

  * Tools invoked in bursts will show **tall bars concentrated in narrow windows**.
  * Regular, consistent usage will appear more evenly distributed.
* Great for validating:

  * **Concurrent execution** patterns
  * **Performance bottlenecks**
  * **Tool scheduling balance**

This is especially useful as you scale into larger datasets, where **temporal structure** of tool usage becomes a key factor in optimization and traceability.


In [ ]:
# 1. Floor timestamps to 60-second bins
df_tool_expanded["time_bin"] = df_tool_expanded["timestamp"].dt.floor("60s")

# 2. Group by time + tool and count
bar_df = (
    df_tool_expanded
    .groupby(["time_bin", "tool_name"])
    .size()
    .reset_index(name="count")
)

# 3. Plot
fig = px.bar(
    bar_df,
    x="time_bin",
    y="count",
    color="tool_name",
    barmode="group",
    text_auto=True,
    title="Tool Invocation Counts per 60 Seconds",
    labels={"time_bin": "Time (1-minute bins)", "count": "Invocation Count"},
    width=1500,
    height=500
)

fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Count",
    margin=dict(t=60, b=80),
    plot_bgcolor='white'
)

fig.show()

### ⏱️ Latency Distribution by Tool (Box Plot with Overlay)

This cell creates a detailed visualization of **latency spread per tool**, combining precise statistical distribution with lightly jittered scatter points to show individual observations.

#### 🧠 What this code does:

* Converts `provider_id` values to human-readable `tool_name`s.
* Parses timestamps and assigns each tool a **numeric x-coordinate** for plotting.
* Adds **horizontal jitter** to each latency point, spreading them slightly to reduce visual overlap.
* Builds two overlaid layers using Plotly:

  1. **Box plots** for each tool’s latency:

     * Shows median, interquartile range, whiskers, and outliers (if any).
     * Styled with consistent violet fills and outlines.
  2. **Scatter trace** of all latency values:

     * Jittered left and right of their corresponding tool box.
     * Semi-transparent for density visibility without visual clutter.

#### 🔍 What you're observing:

* Each box plot shows the **distribution of tool execution times**.
* Points reveal actual latencies, helping identify:

  * Bimodal distributions
  * Outlier clusters
  * Latency consistency (tight boxes with few scattered points)
* Tools with highly variable latencies may require optimization or stability improvements.

This dual-layer view is ideal for **debugging execution bottlenecks**, identifying latency variance across tools, and tracing tool responsiveness under load.


In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from app.enums.tool_enums import ToolProvider

# Step 1: Prep
df_tool_expanded['tool_name'] = df_tool_expanded['provider_id'].apply(ToolProvider.get_name)
df_tool_expanded['timestamp'] = pd.to_datetime(df_tool_expanded['timestamp'])

# Step 2: Assign numeric x positions to categories
tool_names = sorted(df_tool_expanded['tool_name'].unique())
tool_to_num = {tool: i for i, tool in enumerate(tool_names)}
df_tool_expanded['x_numeric'] = df_tool_expanded['tool_name'].map(tool_to_num)

# Step 3: Add jitter to numeric x
jitter_strength = 0.25
df_tool_expanded['x_jittered'] = df_tool_expanded['x_numeric'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_tool_expanded))

# Step 4: Create traces
box_traces = []
for tool, x_val in tool_to_num.items():
    box_traces.append(go.Box(
        y=df_tool_expanded[df_tool_expanded['tool_name'] == tool]['latency_ms'],
        x=[x_val] * len(df_tool_expanded[df_tool_expanded['tool_name'] == tool]),
        name=tool,
        boxpoints=False,
        fillcolor='rgba(120, 81, 169, 0.25)',  # soft violet fill
        line_color='rgba(80, 50, 120, 0.9)',   # strong edge
        marker_color='rgba(80, 50, 120, 0.9)',
        width=0.5,
        showlegend=False
    ))

scatter_trace = go.Scatter(
    x=df_tool_expanded['x_jittered'],
    y=df_tool_expanded['latency_ms'],
    mode='markers',
    marker=dict(
        color='rgba(120, 81, 169, 0.4)',  # softened violet, translucent
        size=4,
    ),
    hoverinfo='skip',
    showlegend=False
)

# Step 5: Build figure
fig = go.Figure(data=box_traces + [scatter_trace])

# Step 6: Layout
fig.update_layout(
    title="Latency Distribution by Tool Provider",
    xaxis=dict(
        tickmode='array',
        tickvals=list(tool_to_num.values()),
        ticktext=list(tool_to_num.keys()),
        tickangle=25,
        title="Tool Provider"
    ),
    yaxis_title="Latency (ms)",
    width=1500,
    height=500,
    plot_bgcolor='white',
    margin=dict(t=60, b=80)
)

fig.show()


### 🧪 Latency Distribution Overlay by Tool

This cell generates a **stacked histogram of latency values**, overlaid by tool, to help compare runtime behavior across tools in a single view.

#### 🧠 What this code does:

* Uses `plotly.express.histogram` to:

  * Plot `latency_ms` for each row in `df_tool_expanded`
  * Color the histogram bars by `tool_name` to distinguish between tools
  * Overlay the distributions instead of stacking or grouping them
  * Add a **rug plot** beneath the x-axis to show raw data density

* Sets bin count to 50 to balance detail and readability.

#### 🔍 What you're observing:

* Each curve shows the **distribution of latency values for a specific tool**.
* Tools with:

  * **Narrow peaks** are more consistent in execution time.
  * **Wider curves** or long tails may have more variability, retries, or overhead.
* Overlaying the tools allows for **direct visual comparison** of latency behavior.

This plot is a fast way to spot anomalies, consistency issues, or performance outliers across your provider population.


In [1]:
import plotly.express as px
px.histogram(
    df_tool_expanded,
    x="latency_ms",
    color="tool_name",
    nbins=50,
    barmode="overlay",
    marginal="rug",
    title="Latency Distribution Overlay by Tool"
)

NameError: name 'df_tool_expanded' is not defined

### 🚨 Average Violations per Tool per Run

This cell computes and visualizes the **mean number of violations** generated by each tool across multiple runs, allowing for direct comparison of compliance or error rates between tools.

#### 🧠 What this code does:

* Adds a new column `violation_count` by parsing the `violations` field:

  * Supports both stringified JSON lists and raw Python lists.
  * Falls back to zero if the field is invalid or empty.

* Aggregates violations in two steps:

  1. **Per run + per tool**, if `run_id` exists (more precise).
  2. Then averages **across all runs per tool** to compute the final metric.

* Constructs a Plotly bar chart with:

  * One bar per tool (sorted ascending by average violation rate).
  * Viridis-reversed color palette for visual consistency with prior plots.
  * Labels on each bar to indicate the average number (rounded to 1 decimal).

#### 🔍 What you're observing:

* This chart shows **which tools tend to produce more violations on average**.
* Tools at the top of the chart are more compliant or robust under test conditions.
* Tools at the bottom may require tuning, rule refinement, or deeper inspection of common failure patterns.
* Especially useful in QA workflows and for identifying validation targets in toolchain optimization.

This view is a key diagnostic for **tool reliability and rule enforcement quality** across sessions.

In [4]:
import json
import plotly.express as px
import pandas as pd
from app.enums.tool_enums import ToolProvider

# Ensure tool_name column exists
df_tool_expanded['tool_name'] = df_tool_expanded['provider_id'].apply(ToolProvider.get_name)

# 1) Compute violation count per row
def count_violations(v):
    try:
        if isinstance(v, str):
            return len(json.loads(v))
        elif isinstance(v, list):
            return len(v)
    except Exception:
        return 0
    return 0

df_tool_expanded['violation_count'] = df_tool_expanded['violations'].apply(count_violations)

# 2) Compute average violations per tool per run
if 'run_id' in df_tool_expanded.columns:
    run_tool = (
        df_tool_expanded
        .groupby(['run_id', 'tool_name'])['violation_count']
        .sum()
        .reset_index()
    )
    avg_violations = (
        run_tool
        .groupby('tool_name')['violation_count']
        .mean()
        .sort_values()
    )
else:
    avg_violations = (
        df_tool_expanded
        .groupby('tool_name')['violation_count']
        .mean()
        .sort_values()
    )

avg_df = avg_violations.reset_index()
avg_df.columns = ['tool_name', 'avg_violations']

# 3) Plotly bar chart
fig = px.bar(
    avg_df,
    x='tool_name',
    y='avg_violations',
    text=avg_df['avg_violations'].round(1),
    color='tool_name',
    color_discrete_sequence=px.colors.sequential.Viridis_r,
    title='Average Violations per Tool per Run',
    labels={
        'tool_name': 'Tool Provider',
        'avg_violations': 'Average Violation Count'
    },
    width=1500,
    height=500
)

# 4) Format bars and layout
fig.update_traces(
    textposition='outside',
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Tool Provider',
    yaxis_title='Average Violation Count',
    showlegend=False,
    plot_bgcolor='white',
    margin=dict(t=60, b=80),
    xaxis_tickangle=25,
)

# 5) Extend y-axis slightly
fig.update_yaxes(range=[0, avg_df['avg_violations'].max() * 1.2])

fig.show()


### ✅ Return Code Distribution and Latency Analysis

This cell generates a **side-by-side visualization** of tool return behavior, showing both outcome frequency and performance spread between success and failure states.

#### 🧠 What this code does:

* Maps numeric `return_code` values into labeled categories: `"Pass (1)"` and `"Fail (0)"`.
* Builds two distinct but aligned subplots using Plotly:

  **Left Plot – Return Code Counts**

  * Counts how many times each return code was observed across all tool executions.
  * Visualizes this with a color-coded bar chart using Set2-style tones.
  * Adds value annotations directly above each bar.

  **Right Plot – Latency by Return Code**

  * Groups latencies by pass/fail status.
  * Displays a boxplot for each group to show median, quartiles, and potential outliers.
  * Uses matching category order and color coding for visual alignment.

#### 🔍 What you're observing:

* The **left chart** gives insight into how often tools succeed vs. fail (`return_code` distribution).
* The **right chart** breaks down how **execution latency** differs for passes vs. failures.
* Often, **failures may cluster with higher or more variable latencies**, indicating retries, timeouts, or degraded performance.
* The fixed category order ensures consistent comparison across plots and avoids visual bias.

This cell is useful when validating return code handling, performance implications of tool errors, and the quality of result codes in FSM processing.

In [5]:
import plotly.graph_objects as go
import pandas as pd

# Define consistent labels and order
label_map = {1: "Pass (1)", 0: "Fail (0)"}
df_tool_expanded["return_code_label"] = df_tool_expanded["return_code"].map(label_map)
order = ["Pass (1)", "Fail (0)"]

# Count data for bar chart
return_counts = (
    df_tool_expanded["return_code_label"]
    .value_counts()
    .reindex(order)
    .fillna(0)
    .astype(int)
)

# 1. Left: Return Code Count Bar Chart
bar_trace = go.Bar(
    x=return_counts.index,
    y=return_counts.values,
    text=return_counts.values,
    textposition="outside",
    marker_color=["#66c2a5", "#fc8d62"],  # Set2 colors
    showlegend=False,
    name="Return Code Counts"
)

# 2. Right: Latency Box Plot
box_traces = []
for code in order:
    subset = df_tool_expanded[df_tool_expanded["return_code_label"] == code]
    box_traces.append(go.Box(
        y=subset["latency_ms"],
        x=[code] * len(subset),
        name=code,
        boxpoints=False,
        marker_color="#66c2a5" if code == "Pass (1)" else "#fc8d62",
        showlegend=False
    ))

# 3. Compose subplots
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Tool Return Code Distribution", "Latency by Return Code"],
    shared_yaxes=False
)

# Add traces
fig.add_trace(bar_trace, row=1, col=1)
for trace in box_traces:
    fig.add_trace(trace, row=1, col=2)

# Update layout
fig.update_layout(
    width=1500,
    height=400,
    showlegend=False,
    title_text=None,
    plot_bgcolor="white",
    margin=dict(t=40, b=60),
)

# Style axes
fig.update_xaxes(
    title_text="Return Code",
    tickangle=0,
    row=1, col=1,
    tickmode="array",
    tickvals=order,
    ticktext=order
)
fig.update_yaxes(title_text="Count", row=1, col=1)

fig.update_xaxes(
    title_text="Return Code",
    tickangle=0,
    row=1, col=2,
    tickmode="array",
    tickvals=order,
    ticktext=order
)
fig.update_yaxes(title_text="Latency (ms)", row=1, col=2)

fig.show()


### 📊 Summary Message Frequency

This cell builds and visualizes a **bar chart of tool-level summary message frequencies**, helping identify which outcomes or end states are most common in tool outputs.

#### 🧠 What this code does:

* Counts the frequency of each distinct `summary` value in the `df_tool_expanded` DataFrame.
* Sorts these summaries in ascending order to group lower-frequency events on the left.
* Constructs a Plotly bar chart with:

  * Categorical x-axis for each summary message (supports emoji, text, etc.)
  * Bar heights proportional to message count
  * Automatic color differentiation via the `Magma_r` palette
  * Annotations on each bar showing the exact count

#### 🔍 What you're observing:

* This chart shows **how often each type of summary message appears**, across all tools and runs.
* Frequently recurring summaries may indicate common system paths, defaults, or fallback states.
* Rare summaries may indicate exceptional paths, errors, or specific edge behaviors.
* Helps validate that summaries are properly categorized, and can surface unexpected tool states.

This view is especially useful when analyzing tool convergence behavior, failure rates, or standardization compliance across summary types.


In [6]:
import plotly.express as px
import pandas as pd

# 1) Build summary count DataFrame
summary_counts = df_tool_expanded["summary"].value_counts(dropna=False).sort_values()
summary_df = summary_counts.reset_index()
summary_df.columns = ['summary', 'count']

# 2) Plotly bar chart
fig = px.bar(
    summary_df,
    x='summary',
    y='count',
    color='summary',
    color_discrete_sequence=px.colors.sequential.Magma_r,
    text=summary_df['count'],
    title="Summary Message Frequency",
    labels={
        'summary': 'Summary',
        'count': 'Count'
    },
    width=1500,
    height=500
)

# 3) Layout + annotation styling
fig.update_traces(
    textposition='outside',
    marker_line_color='black',
    marker_line_width=1
)

fig.update_layout(
    xaxis_title='Summary',
    yaxis_title='Count',
    showlegend=False,
    plot_bgcolor='white',
    margin=dict(t=60, b=80),
    xaxis=dict(tickangle=25),
)

# 4) Extend y-axis range slightly for spacing
fig.update_yaxes(range=[0, summary_df['count'].max() * 1.2])

fig.show()
